In [ ]:
# from vllm import LLM, SamplingParams
# model_name = "Qwen/Qwen2-VL-2B-Instruct"
# llm = LLM(model=model_name)

In [1]:
import sys
sys.path.append("data/g3")
from dataset import MsgPackIterableDatasetMultiTargetWithDynLabels

In [20]:
# image, problem, solution
import json
target_mapping = json.load(open("data/g3/val/countries.json"))
ds = iter(MsgPackIterableDatasetMultiTargetWithDynLabels(
    path="data/g3/val/msgpack",
    target_mapping=target_mapping,
    key_img_id="id",
    key_img_encoded="image",
    shuffle=False,
    cache_size=350000,
))

In [22]:
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import io
import base64
from datasets import Image

def list_to_dict(list_of_dicts):
    result = {}
    for d in list_of_dicts:
        for key, value in d.items():
            result.setdefault(key, []).append(value)
    return result

problem = "Based on visual cues, such as road signs, architecture, vegetation, and landscape, identify the most likely country where this image was taken."
country_to_id = pd.read_csv("data/g3/countries.csv", skiprows=0)
country_to_id = {c["class_label"]: c["country"] for c in country_to_id.to_dict(orient="records")}
hf = []
counter = 0
for x in tqdm(ds):
    ann = {"image": x[0], "id": x[2], "solution": f"<answer>{country_to_id[x[1]]}</answer>", "problem": problem}
    hf.append(ann)
    counter += 1
    # if counter > 10:
    #     break

0it [00:00, ?it/s]

3887it [00:00, 22407.24it/s]


In [13]:
import numpy as np
idxs = np.random.permutation(range(len(hf)))

In [23]:
# chunk = [hf[i] for i in idxs[:10000]]
chunk = hf
chunk = list_to_dict(chunk)
chunk = Dataset.from_dict(chunk)
chunk = chunk.cast_column("image", Image())
chunk.to_parquet(f"data/g3/data/val.parquet")

Creating parquet from Arrow format: 100%|██████████| 39/39 [00:00<00:00, 201.19ba/s]


87890429

In [ ]:
chunk_size = 50000
n_chunks = (len(hf) + chunk_size - 1) // chunk_size  # Calculate number of chunks

for i in range(n_chunks):
    print(i)
    start = i * chunk_size
    end = min(start + chunk_size, len(hf))
    chunk = hf[start:end]
    chunk = list_to_dict(chunk)
    chunk = Dataset.from_dict(chunk)
    chunk = chunk.cast_column("image", Image())
    chunk.to_parquet(f"data/g3/data/train_{str(i).zfill(3)}.parquet")

In [27]:
import datasets
x = datasets.load_dataset("data/g3/g3-streetview-10k/data")

In [28]:
x.shuffle(42)

DatasetDict({
    train: Dataset({
        features: ['image', 'id', 'solution', 'problem'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['image', 'id', 'solution', 'problem'],
        num_rows: 3887
    })
})

In [25]:
x["train"][0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=426x320>,
 'id': 'Rq68On8-AH5THfBgKBrVrQ_0.png',
 'solution': '<answer>Brazil</answer>',
 'problem': 'Based on visual cues, such as road signs, architecture, vegetation, and landscape, identify the most likely country where this image was taken.'}